In [1]:
import pandas as pd
import numpy as np
import boto3
from io import BytesIO

s3 = boto3.client(
    "s3",
    endpoint_url="http://minio:9000",
    aws_access_key_id="minioadmin",
    aws_secret_access_key="minioadmin123",
)
BUCKET = "oil-pipeline"

def read_parquet(key):
    obj = s3.get_object(Bucket=BUCKET, Key=key)
    return pd.read_parquet(BytesIO(obj["Body"].read()))

def save_parquet(df, key):
    buf = BytesIO()
    df.to_parquet(buf, index=False)
    buf.seek(0)
    s3.put_object(Bucket=BUCKET, Key=key, Body=buf.getvalue())
    print(f"Saved: {key} ({len(df)} rows)")

In [2]:
deliveries = read_parquet("cleaned/deliveries/data.parquet")
drivers = read_parquet("cleaned/drivers/drivers.parquet")
vehicles = read_parquet("cleaned/vehicles/vehicles.parquet")

deliveries["date"] = pd.to_datetime(deliveries["date"])
print(deliveries.shape)
deliveries.head()

(30, 12)


,delivery_id,date,source,destination,product_type,volume_ton,cost_usd,delay_hours,distance_km,weather_conditions,driver_id,vehicle_id
0,1,2025-10-01,Base-Khanty,Station-01,Diesel,32.5,2100.50,0.0,180.0,Clear,1,1
1,2,2025-10-01,Base-Tomsk,Station-02,Gasoline,28.0,1850.00,1.5,150.0,Rain,2,2
2,3,2025-10-01,Base-Tyumen,Station-03,Diesel,22.0,1650.25,0.0,120.0,Clear,3,3
3,4,2025-10-02,Base-Khanty,Station-04,Kerosene,35.0,2400.80,2.0,210.0,Fog,4,1
4,5,2025-10-02,Base-Tomsk,Station-05,Gasoline,20.5,1500.00,0.5,110.0,Cloudy,5,2


In [3]:
# задержки по погоде
print("Средняя задержка по погоде:")
weather_delay = deliveries.groupby("weather_conditions")["delay_hours"].mean().sort_values(ascending=False)
print(weather_delay.round(2))

# задержки по расстоянию
deliveries["distance_bucket"] = pd.cut(
    deliveries["distance_km"],
    bins=[0, 120, 160, 220],
    labels=["short (<120km)", "medium (120-160km)", "long (>160km)"]
)
print("\nСредняя задержка по расстоянию:")
print(deliveries.groupby("distance_bucket")["delay_hours"].mean().round(2))

Средняя задержка по погоде:
weather_conditions
Snow      3.50
Rain      2.00
Fog       1.50
Cloudy    0.38
Clear     0.10
Name: delay_hours, dtype: float64

Средняя задержка по расстоянию:
distance_bucket
short (<120km)        1.94
medium (120-160km)    0.69
long (>160km)         0.39
Name: delay_hours, dtype: float64


/tmp/ipykernel_12262/2759384118.py:13: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(deliveries.groupby("distance_bucket")["delay_hours"].mean().round(2))


In [4]:
deliveries["cost_per_km"] = (deliveries["cost_usd"] / deliveries["distance_km"]).round(2)

print("Средний cost/km по типу продукта:")
print(deliveries.groupby("product_type")["cost_per_km"].mean().round(2))

print("\nСредний cost/km по маршруту источника:")
print(deliveries.groupby("source")["cost_per_km"].mean().sort_values(ascending=False).round(2))

Средний cost/km по типу продукта:
product_type
Diesel      12.76
Gasoline    12.13
Kerosene    13.03
Name: cost_per_km, dtype: float64

Средний cost/km по маршруту источника:
source
Base-Omsk      13.58
Base-Tyumen    13.26
Base-Tomsk     12.47
Base-Khanty    11.74
Name: cost_per_km, dtype: float64


In [5]:
driver_kpi = deliveries.groupby("driver_id").agg(
    total_deliveries=("delivery_id", "count"),
    avg_delay=("delay_hours", "mean"),
    total_volume=("volume_ton", "sum"),
    avg_cost_per_km=("cost_per_km", "mean"),
    total_distance=("distance_km", "sum"),
).reset_index()

driver_kpi = driver_kpi.merge(drivers[["driver_id", "name", "experience_years", "region"]], on="driver_id")
driver_kpi = driver_kpi.sort_values("avg_delay")

print(driver_kpi[["name", "experience_years", "total_deliveries", "avg_delay", "total_volume"]].to_string(index=False))
save_parquet(driver_kpi, "marts/mart_driver_kpi.parquet")

            name  experience_years  total_deliveries  avg_delay  total_volume
     Ivan Petrov                 8                 5   0.200000         168.4
 Aleksey Smirnov                 5                 7   0.428571         165.1
    Pavel Egorov                15                 4   0.625000         130.7
  Sergey Sidorov                12                 7   0.785714         195.2
Andrey Kuznetsov                10                 7   2.285714         141.1
Saved: marts/mart_driver_kpi.parquet (5 rows)


In [6]:
mart_delays = deliveries[[
    "delivery_id", "date", "source", "destination",
    "product_type", "volume_ton", "cost_usd", "cost_per_km",
    "delay_hours", "distance_km", "weather_conditions",
    "driver_id", "vehicle_id"
]].copy()

save_parquet(mart_delays, "marts/mart_logistics.parquet")
print("Done!")

Saved: marts/mart_logistics.parquet (30 rows)
Done!
